# PragyanAI – YOLO26n ExecuTorch Export

Run this notebook in Google Colab to download the official YOLO26n weights, export them to ExecuTorch, test inference, and download the model folder.

**Expected output:** `yolo26n_executorch_model/model.pte` and `metadata.yaml`.

The `.pte` is generated by Ultralytics during export. It is not included as a fake placeholder in the source ZIP.

In [ ]:
# 1. Install Ultralytics
!pip -q install --upgrade ultralytics


In [ ]:
# 2. Download/load official YOLO26n PyTorch weights and export to ExecuTorch
from ultralytics import YOLO
from pathlib import Path

model = YOLO("yolo26n.pt")
export_path = model.export(
    format="executorch",
    imgsz=640,
    batch=1,
    device="cpu"
)
print("Export returned:", export_path)

model_dir = Path("yolo26n_executorch_model")
assert (model_dir / "model.pte").is_file(), "model.pte was not created"
assert (model_dir / "metadata.yaml").is_file(), "metadata.yaml was not created"
print("Model directory ready:", model_dir.resolve())
print("PTE size (MB):", round((model_dir / "model.pte").stat().st_size / (1024*1024), 2))


In [ ]:
# 3. Smoke-test the exported ExecuTorch model
exported_model = YOLO(str(model_dir))
results = exported_model("https://ultralytics.com/images/bus.jpg", imgsz=640, verbose=False)
print("Inference succeeded.")
print("Detection count:", len(results[0].boxes) if results[0].boxes is not None else 0)


In [ ]:
# 4. Package the model directory for backend/models/
import shutil
from pathlib import Path

deployment_model_dir = Path("backend_models") / "yolo26n_executorch_model"
deployment_model_dir.parent.mkdir(parents=True, exist_ok=True)
if deployment_model_dir.exists():
    shutil.rmtree(deployment_model_dir)
shutil.copytree(model_dir, deployment_model_dir)

zip_base = "PragyanAI_YOLO26n_ExecuTorch_Model"
shutil.make_archive(zip_base, "zip", root_dir="backend_models")
print("Created:", zip_base + ".zip")


In [ ]:
# 5. Download model artifact ZIP
from google.colab import files
files.download("PragyanAI_YOLO26n_ExecuTorch_Model.zip")


## Upload location in GitHub

Extract the downloaded model ZIP and copy its folder into:

```text
backend/
  models/
    yolo26n_executorch_model/
      model.pte
      metadata.yaml
```

On Render, set:

```env
MODEL_PATH=models/yolo26n_executorch_model
```

If `model.pte` exceeds GitHub's ordinary 100 MB file limit, use Git LFS or external object storage and download it during deployment. Never rename `yolo26n.pt` to `model.pte`.